# Transformer Translation Draft

Notebook nay phan tich nhanh EVBCorpus SGML va ghi lai baseline Transformer translation truoc khi refactor vao `src/`.

In [ ]:
from pathlib import Path
import html
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CORPUS_DIR = PROJECT_ROOT / 'data' / 'EVBCorpus_EVBNews_v2.0'
files = sorted(CORPUS_DIR.glob('*.sgml'))
len(files), files[:3]

In [ ]:
def extract_lang(text: str, lang: str):
    pattern = re.compile(rf"<s\\s+id=['\"]{lang}(\\d+)['\"]>(.*?)</s>", re.IGNORECASE | re.DOTALL)
    return {int(m.group(1)): html.unescape(re.sub(r'\\s+', ' ', m.group(2)).strip()) for m in pattern.finditer(text)}

text = files[0].read_text(encoding='utf-8', errors='ignore')
en = extract_lang(text, 'en')
vn = extract_lang(text, 'vn')
pairs = [{'en': en[i], 'vn': vn[i]} for i in sorted(en) if i in vn]
pd.DataFrame(pairs).head()

In [ ]:
sample = pd.DataFrame(pairs)
pd.DataFrame({
    'en_tokens': sample['en'].str.split().str.len(),
    'vn_tokens': sample['vn'].str.split().str.len(),
}).describe()

## Baseline idea

Pipeline trong `src/` dung PyTorch `nn.Transformer`:

```text
English sentence -> source vocab -> Transformer encoder
Vietnamese sentence -> target vocab -> causal Transformer decoder -> next token
```

Quality duoc report bang validation loss, sample BLEU va bang source/reference/prediction.

In [ ]:
# Quick smoke test from week4/Transformer:
# !python src/main.py --epochs 1 --max-pairs 1000 --batch-size 16 --device cpu

In [ ]:
history_path = PROJECT_ROOT / 'output' / 'history.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history)
    history[['train_loss', 'valid_loss']].plot(figsize=(8, 4), title='Transformer training history')
else:
    print('Run python src/main.py first to generate output/history.csv')

In [ ]:
report_path = PROJECT_ROOT / 'output' / 'evaluation_report.json'
samples_path = PROJECT_ROOT / 'output' / 'translation_samples.md'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8'))
if samples_path.exists():
    print(samples_path.read_text(encoding='utf-8')[:3000])